# Assignment 1 — the order flow imbalance on INTC

Federico Andrione

Points i.–v. of Assignment 1 (Section 1.5 of the notes), carried through on one ticker and
one window of one trading day.

| | |
| --- | --- |
| ticker | INTC |
| day | 21 June 2012 |
| window | 10:00–13:00 |
| reported depth | 10 levels |
| tick size $\tau$ | \$0.01 |

The window leaves out the open and the close.

**How to read it:** Each point opens with what the assignment asks and what is found in this
window. The cells below it show where each number comes from. The Summary at the end collects
the five answers.

**Where the code is:** The reusable code is in `unito26/lob/Federico_Andrione_reconciliation.py`,
tested in `tests/lob/test_Federico_Andrione_reconciliation.py`. The data is read from
`data/lobster`, which is not in the repository.

**Setup:** The imports, the data and the window, a helper that writes numbers as they are read,
and the style of the figures:

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

from unito26.lob import estimation as est, imbalance_regression as ir, lobster
from unito26.lob.Federico_Andrione_reconciliation import (
    as_book_message, discrepancy_census, reconcile, repair_log, replay,
)
from unito26.lob.lobster import LOBSTER_UNITS_PER_DOLLAR, LobsterEvent
from unito26.lob.lobster_session import (
    LobsterMarketSession, coarsening_report, coarsening_totals,
)
from unito26.lob.messages import (
    BUY, SELL, GridDepth, Horizon, MessageType, TickGrid, Window,
)
from unito26.lob.orderbook import AggregateBook
from unito26.lob.statistics import SessionStatistics

DATA = Path("..") / "data" / "lobster"
CENT = TickGrid(0.01)
SPEC = SessionStatistics((GridDepth(1),), (), (1, 10, 60))

INTC = lobster.LobsterFiles.parse(DATA / "INTC_2012-06-21_34200000_57600000_message_10.csv")
WINDOW = lobster.TradingWindow(36000.0, 46800.0)


def readable(value):
    '''A number as it is read: whole numbers with separators, others to two decimals.'''
    return f"{value:,.0f}" if float(value).is_integer() else f"{value:,.2f}"


# The colours of the figures: one for each series, and two for text.
BLUE, ORANGE = "#2a78d6", "#eb6834"
LIGHT_BLUE, DARK_BLUE = "#86b6ef", "#104281"
INK, MUTED = "#0b0b0b", "#52514e"


def axes(title, xlabel, ylabel):
    """One pair of axes in the style of this notebook: a title, labelled axes, a light grid."""
    _, ax = plt.subplots(figsize=(8, 3.6))
    ax.set_title(title, loc="left", fontsize=11, color=INK)
    ax.set_xlabel(xlabel, color=MUTED)
    ax.set_ylabel(ylabel, color=MUTED)
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", color="#e6e5e1", linewidth=0.8)
    ax.set_axisbelow(True)
    ax.tick_params(colors=MUTED)
    return ax

---

## i. The session, and the reconciliation of the book

> **Asked.** Build the session from the message and orderbook files; reconcile the rebuilt
> book with the orderbook file LOBSTER supplies; account for the discrepancies.
>
> **Found.** Rebuilt from the messages alone, the book agrees with the file on all ten levels
> for the first 63 rows only, and at the first level on 59.58% of the rows. In this window
> every difference the comparison can see comes from the file showing ten levels only: with a
> repair that reads 579 levels from the file, every row agrees.

### i.1 Building the session

LOBSTER supplies two files for the day.

- The **message file** has one row per event: an order submitted, cancelled or executed.
- The **orderbook file** has, after each event, the ten best prices on each side, the
  *levels*, with the size resting at each.

Row $k$ of the second is the book after message $k$; nothing else links them, so the two must
have the same number of rows.

The orderbook file is the configuration of Section 1.1 truncated at $L = 10$ levels (Section 1.2.1),

$$\big(P^a_t,\ P^b_t,\ \{(S^{a,i}_t, S^{b,i}_t) : i \le L\}\big).$$

The first check is that the two files are aligned, row for row:

In [ ]:
census = lobster.file_census([INTC]).set_index("Ticker")
pd.DataFrame({
    "message rows": census["MessageRows"],
    "orderbook rows": census["BookRows"],
    "aligned": census["MessageRows"] == census["BookRows"],
})

The two files have the same number of rows, so they can be read side by side.

The window is cut in the code, not in the files: `LobsterMarketSession.from_files` keeps the
rows between 10:00 and 13:00, and checks that every price is a whole number of cents. What the
window holds, by type of message:

In [ ]:
raw = LobsterMarketSession.from_files(INTC, SPEC, CENT, WINDOW)
# The aligned pair and its units, as reconcile, replay and repair_log take them.
PAIR = (raw.messages, raw.book, raw.price_unit, raw.reported_depth)
events = (
    raw.messages["Type"].value_counts()
    .reindex([int(event) for event in LobsterEvent], fill_value=0)
)
pd.DataFrame(
    {"messages": events.to_numpy(), "share": (events / events.sum()).map("{:.1%}".format).to_numpy()},
    index=pd.Index([event.name.replace("_", " ").lower() for event in LobsterEvent], name="event"),
)

Submissions and deletions make up almost all the messages; visible executions are 4.1% of
them and hidden ones 0.4%, and there is no trading halt or cross trade.

**One row per order that trades:** The files write one row for each *resting* order that is
executed. So an *aggressive* order, one that arrives and trades at once, takes several rows
when it hits several resting orders: for example, a buy that meets three sell orders is three
rows.

Points ii.–v. count events, and that buy is one event. `coarsened` therefore merges its rows
into the last one and adds up its fills (Section 1.2.5.1). The result, `session`, is the
session that points ii.–v. work on.

The table compares the session before and after the merging, to check that nothing was lost,
both what must be conserved and what must change:

In [ ]:
session = raw.coarsened(True)
totals = coarsening_totals(raw, session).set_index("Quantity")[["Fine", "Coarse"]]
in_ticks = ["traded value in tick-shares", "VWAP in ticks"]
totals.loc[in_ticks] = totals.loc[in_ticks] * CENT.tick_size
totals.rename(
    index={"traded value in tick-shares": "traded value ($)", "VWAP in ticks": "VWAP ($)"},
    columns={"Fine": "a row per execution", "Coarse": "a row per aggressive order"},
).apply(lambda column: column.map(readable))

Shares, traded value and VWAP are the same before and after: the merging only regroups
trades, so none of them may change. The number of rows falls and the mean size of what trades rises,
because a row that trades is now a whole aggressive order rather than one of its fills.
What the merging does to $e_n$ is measured in ii.1.

### i.2 Reconciling the rebuilt book with the orderbook file

**The rebuild** is the fold

$$\mathcal B_k = F(\mathcal B_{k-1}, m_k). \tag{1.2.1}$$

- $\mathcal B$ is held in the course's `AggregateBook`, which keeps the total size resting at
  each price.
- $F$ is its `apply`, and $m_k$ the $k$-th message.
- It starts from the first row of the orderbook file rather than from
  $\mathcal B_0 = \varnothing$, since the messages refer to orders submitted before the window
  opened.

Each message changes the book as follows. The side is the one `Direction` names, which for an
execution is the side of the resting order (Remark 1.2.5).

| message type | effect on the rebuilt book |
| --- | --- |
| 1, submission | adds its size at its price, after matching it against the opposite side |
| 2, 3, 4: partial cancellation, deletion, visible execution | remove its size at its price |
| 5, 6, 7: hidden execution, cross trade, trading halt | none |

**The comparison:** After each message `reconcile` compares the rebuilt book with the file,
level by level, and records the first level at which they differ, or 0 where they agree. This
is the check of Remark 1.2.3: the orderbook file is the exchange's own fold of the same
messages, so a difference points to an error in the rebuild, or to something the messages do
not report.

The cell rebuilds the book from the messages alone (its last argument, `False`: nothing is
taken from the file after the first row) and prints how long the two books stay identical:

In [ ]:
alone = reconcile(*PAIR, False)
first = int(alone.loc[alone["FirstDifferingLevel"] > 0, "Row"].min())
later = alone[alone["Row"] >= first]
print(f"The first {first - 1} rows agree on all ten levels. From row {first} on, "
      f"{(later['FirstDifferingLevel'] == 0).sum():,} of the remaining {len(later):,} do.")

The two books agree on all ten levels for the first 63 rows, and on none after. No later
message says anything about a level the rebuild got wrong, so no message corrects it, and
further errors add to it. Only the matching can remove it, when a submission crosses it (end of
i.2).

Not every error reaches the best prices, though. The table counts the rows that agree in two
ways. Both books have one row per message, and a row **agrees** when, after that message, the
rebuilt book holds the same numbers as the file.

- *All ten levels* compares the 40 numbers of the row: price and size of ten levels on each
  side.
- *First level only* compares the 4 numbers of the **touch**: price and size of the best bid,
  the highest price at which someone is buying, and the price and size of the best ask, the lowest price at which
  someone is selling. The touch is what $P^m$, $I^1$ and $\mathrm{OFI}$ read.

The rows that agree, in the two ways:

In [ ]:
def agreement(report):
    differing = report["FirstDifferingLevel"]
    agreeing = {
        "all ten levels": int((differing == 0).sum()),
        "first level only": int((differing != 1).sum()),
    }
    return pd.Series({
        comparison: f"{count:,} of {len(report):,} ({count / len(report):.2%})"
        for comparison, count in agreeing.items()
    })


agreement(alone).rename("rows that agree").rename_axis("comparison").to_frame()

All ten levels agree on almost no row, the first level on 59.58% of them: most errors stay
below the touch. 

An error reaches it by rising to it: as the levels above empty one by one, a level the rebuild holds at the wrong
size, or does not hold at all, moves up to the first place.


**A submission that trades:** In the market a submission never trades: an incoming order that
trades appears as the executions of the resting orders it hits, and only its remainder, if
any, as a submission. The rebuild does not take that for granted. `apply` treats a submission
as Proposition 1.1.11 treats an incoming order, matching it against the opposite side before
resting what is left.

On a book that agrees with the file nothing matches. On one that has gone wrong a submission
can cross a level that should not be there, and trade against it.

The cell counts the submissions that trade in the replay from the messages alone, and the
shares they take:

In [ ]:
def submissions_traded(refill):
    """Submissions that trade in the replay of `reconcile`, and the shares they take."""
    taken = [book.last_traded_size for _, _, kind, _, _, book, _ in replay(*PAIR, refill)
             if kind == LobsterEvent.SUBMISSION and book.last_traded_size > 0]
    return len(taken), sum(taken)


traded, taken = submissions_traded(False)
print(f"submissions that trade, messages alone: {traded:,}, taking {taken:,} shares")

**The matching:** In the replay from the messages alone, 188 submissions trade: each crossed
a level the rebuild held and the file did not, and took size from it. These are not trades of
the market.

To see what that matching does for the touch, the cell replays again without it, the size of
each message simply added or removed at its price (`set_size` in place of `apply`), and prints
the first-level agreement with and without the matching:

In [ ]:
def first_level_agreement_without_matching():
    """Rows whose first level agrees with the file when each message's size is added or
    removed at its price, with no matching."""
    book = AggregateBook.from_lobster_row(raw.book.iloc[0], raw.price_unit, raw.reported_depth)
    touches = raw.book[["BidPrice1", "BidSize1", "AskPrice1", "AskSize1"]].to_numpy()
    agree = 0
    for row, message in enumerate(raw.messages.iloc[1:].itertuples(), start=1):
        step = as_book_message(message.Type, message.Time, message.Size, message.Price,
                               message.Direction, raw.price_unit)
        if step is not None:
            change = step.size if step.kind is MessageType.SUBMIT else -step.size
            held = book.size_at(step.direction, step.price)
            book.set_size(step.direction, step.price, max(held + change, 0))
        bid, ask = book.best_price(BUY), book.best_price(SELL)
        mine = (bid * raw.price_unit, book.size_at(BUY, bid),
                ask * raw.price_unit, book.size_at(SELL, ask))
        agree += mine == tuple(touches[row])
    return agree


with_matching = int((alone["FirstDifferingLevel"] != 1).sum())
without_matching = first_level_agreement_without_matching()
print(f"first level agrees, with matching:      {with_matching:,} of {len(alone):,} "
      f"({with_matching / len(alone):.2%})")
print(f"first level agrees, without matching:   {without_matching:,} of {len(alone):,} "
      f"({without_matching / len(alone):.2%})")

Without the matching the first level agrees on 10.67% of the rows instead of 59.58%. So the
matching corrects part of what goes wrong: it removes levels the rebuild should not hold, and
the first level agrees on far more rows with it than without.

The trades it makes are not trades of the market, and with the repair of i.3 there are none.

### i.3 Accounting for the discrepancies

The message file reports only what happens inside the ten levels the orderbook file shows.
That truncation has **two** effects:

**1) A level enters from below:** When a level empties, every level beyond it moves up one place
and a new tenth level comes into view. If no message has mentioned its price, the rebuilt book
has nothing to put there.

The cell prints the message of the first row that differs, and the levels 1, 2, 9 and 10 of
the side where it differs, in the file and in the rebuilt book:

In [ ]:
def shown_levels(book, direction, levels):
    held = book.occupied_levels(direction, raw.reported_depth)
    return [
        f"${held[level - 1][0] * CENT.tick_size:.2f} × {held[level - 1][1]:,}"
        if level <= len(held) else "no level"
        for level in levels
    ]


message = raw.messages.iloc[first]
event = LobsterEvent(int(message["Type"])).name.replace("_", " ").lower()
print(f"row {first}: {event} of {int(message['Size'])} shares "
      f"at ${message['Price'] / LOBSTER_UNITS_PER_DOLLAR:.2f}")

in_file = AggregateBook.from_lobster_row(raw.book.iloc[first], raw.price_unit,
                                         raw.reported_depth)
# The book rebuilt from the messages alone, as the replay holds it after message `first`.
rebuilt = next(book for row, *_, book, _ in replay(*PAIR, False) if row == first)
direction = next(
    side for side in (BUY, SELL)
    if in_file.occupied_levels(side, raw.reported_depth)
    != rebuilt.occupied_levels(side, raw.reported_depth)
)
levels = (1, 2, raw.reported_depth - 1, raw.reported_depth)
pd.DataFrame(
    {"orderbook file": shown_levels(in_file, direction, levels),
     "rebuilt": shown_levels(rebuilt, direction, levels)},
    index=pd.Index(levels, name=f"{'bid' if direction == BUY else 'ask'} level"),
)

Row 64 is a visible execution of 100 shares. The levels move up, and the file's new tenth
level, \$27.24, is one the rebuilt book does not have: in this window the first difference is
of this kind.

**2) A level leaves the view and returns:** A new order at a better price pushes the tenth level
to the eleventh, where nothing is reported. If that level changes there and later returns,
the rebuilt book still holds its old size. The tests check it on a hand-made example
(`test_a_level_returning_into_view_is_read_again_from_the_file`); it also happens in this window.

The cell finds the first level that left the view and came back with a different size, and
shows, at the row where it returns, the levels 1, 2, 9 and 10 of its side, in the file and in
the book rebuilt from the messages alone:

In [ ]:
log = repair_log(*PAIR)
forgotten = (
    log[log["Action"] == "forget"]
    .rename(columns={"Row": "RowForgotten", "Size": "SizeWhenForgotten"})
    [["RowForgotten", "Direction", "Price", "SizeWhenForgotten"]]
)
imported = log[log["Action"] == "import"][["Row", "Direction", "Price", "Size"]]
# Each import set against the last time the same level was forgotten, if it ever was.
taken = pd.merge_asof(
    imported, forgotten, left_on="Row", right_on="RowForgotten",
    by=["Direction", "Price"], allow_exact_matches=False,
)
returning = taken["RowForgotten"].notna()
changed = returning & (taken["Size"] != taken["SizeWhenForgotten"])

back = taken[changed].iloc[0]
left, row, side = int(back["RowForgotten"]), int(back["Row"]), int(back["Direction"])
name = "bid" if side == BUY else "ask"
print(f"row {left}: the {name} at ${back['Price'] * CENT.tick_size:.2f} leaves the view, "
      f"with {int(back['SizeWhenForgotten']):,} shares")
print(f"row {row}: it is back in view")

in_file = AggregateBook.from_lobster_row(raw.book.iloc[row], raw.price_unit, raw.reported_depth)
rebuilt = next(book for r, *_, book, _ in replay(*PAIR, False) if r == row)
place = [price for price, _ in in_file.occupied_levels(side, raw.reported_depth)].index(
    int(back["Price"])) + 1
levels = (1, 2, place - 1, place)
pd.DataFrame(
    {"orderbook file": shown_levels(in_file, side, levels),
     "rebuilt": shown_levels(rebuilt, side, levels)},
    index=pd.Index(levels, name=f"{name} level"),
)

The ask at \$27.40 left the view at row 628 with 3,772 shares and is back at row 676, where the
file shows 2,972. No message reported the change, so the rebuild from the messages alone still
holds 3,772: the levels above agree, the tenth has the right price and the wrong size.

**The repair:** Both effects come from levels moving across the tenth place, and the repair
acts there, in two steps after every message.

- *Keep ten levels.* The rebuilt book drops whatever lies beyond its tenth level on each
  side. A level that leaves the view is forgotten, rather than kept at a size no message
  will update. Nothing is copied in this step.
- *Fill from below.* A side left with fewer than ten levels takes from the file the levels
  beyond its deepest one, with the size the file shows. This supplies a level that enters
  from below for the first time, and a level that returns after it was forgotten.

So a level entering from below is repaired by the second step alone. A level that leaves the
view and returns needs both: the first to forget it as it leaves, the second to read it again
as it returns.

A price the rebuilt book holds within the ten levels is never copied: it is compared, and a
wrong size there is still a difference.

The cell reconciles again with the repair (last argument `True`) and sets the rows that agree
beside those of i.2:

In [ ]:
repaired = reconcile(*PAIR, True)
pd.DataFrame({
    "messages alone": agreement(alone),
    "with the repair": agreement(repaired),
}).rename_axis("comparison")

With the repair every row agrees, on all ten levels.

Two checks on that agreement. If the copying made it, the rows where nothing was copied would
still differ; and on a correct book no submission should trade. The cell counts the
submissions that trade in the repaired replay, and the rows that agree among those where
nothing was imported:

In [ ]:
print(f"submissions that trade, with the repair: {submissions_traded(True)[0]:,}")
clean = repaired[repaired["LevelsImported"] == 0]
agreement(clean).rename("rows that agree, nothing imported").rename_axis("comparison").to_frame()

No submission trades in the repaired replay, as none trades in the market. Rows where nothing
was copied from the file agree too, so the agreement is not made by the copying: in this
window, no other cause produces a difference the comparison can see.

The repair copies little from the file, and only where the messages cannot know: after an
event that empties a level, a deletion or a visible execution. The cell counts the levels taken
from the file, and the events after which they were taken:

In [ ]:
imports = discrepancy_census(repaired).set_index("Event")["LevelsImported"]
rows_importing = int((repaired["LevelsImported"] > 0).sum())
print(f"levels taken from the file: {imports.sum():,}, in {rows_importing:,} of {len(repaired):,} rows "
      f"({rows_importing / len(repaired):.2%})")
print(f"  after a deletion:          {imports['DELETION']:>5,}")
print(f"  after a visible execution: {imports['EXECUTION_VISIBLE']:>5,}")
print(f"  after any other event:     {imports.drop(['DELETION', 'EXECUTION_VISIBLE']).sum():>5,}")

579 levels are taken from the file, in 0.21% of the rows, all after a deletion or a visible
execution.

They are of two kinds. One imported at a price the repair forgot earlier, on the same side, is
a level returning into view; any other enters for the first time. A returning level whose size
changed while out of view is an error the messages alone would have carried. The cell sorts the
imports into the two kinds:

In [ ]:
pd.Series({
    "entering from below for the first time": int((~returning).sum()),
    "returning after leaving the view": int(returning.sum()),
    "  of which with a size changed out of view": int(changed.sum()),
}).rename("levels imported").to_frame()

In this window most of the levels imported return after leaving the view, and most of those
changed while out of it. Leaving the view and returning is the larger of the two effects.

**Other known pitfalls:** The course's `unito26.lob.lobster` lists the features of LOBSTER data
that make a rebuild go wrong. For each, the table counts the rows of the window where it
occurs:

In [ ]:
messages = raw.messages
kind = messages["Type"].to_numpy()
stamps = messages["TimeNanoseconds"].to_numpy()
submitted = messages.loc[kind == LobsterEvent.SUBMISSION, "OrderID"]
removing = np.isin(kind, [
    LobsterEvent.PARTIAL_CANCELLATION, LobsterEvent.DELETION, LobsterEvent.EXECUTION_VISIBLE,
])
orders = raw.market_orders()
fills = np.bincount(orders[orders >= 0])

where = {
    "asymmetric reporting: a visible execution": kind == LobsterEvent.EXECUTION_VISIBLE,
    "halts and cross trades": np.isin(kind, [LobsterEvent.CROSS_TRADE, LobsterEvent.TRADING_HALT]),
    "hidden liquidity: a hidden execution": kind == LobsterEvent.EXECUTION_HIDDEN,
    "pre-existing orders: removal of an order submitted before the window":
        removing & ~messages["OrderID"].isin(submitted).to_numpy(),
    "repeated timestamps: same instant as the previous message":
        np.r_[False, np.diff(stamps) == 0],
    "execution granularity: one of several fills of one order":
        (orders >= 0) & (fills[np.maximum(orders, 0)] > 1),
}
replayed = np.r_[False, np.ones(len(repaired), dtype=bool)]

pd.DataFrame(
    {"rows where it occurs": [int((rows & replayed).sum()) for rows in where.values()]},
    index=pd.Index(list(where), name="pitfall"),
)

Every pitfall occurs in this window except halts and cross trades. With the repair every row
agrees, so none of them causes a difference here; the list below says why.

- *Asymmetric reporting.* For an execution, `Direction` is the side of the resting order,
  not of the order that traded against it. Read the other way, as the side of the aggressor,
  the rebuild removes size from the wrong side at every visible execution, and the rows that
  follow inherit the error.
- *Halts and cross trades.* Special messages whose `Price` is not a price. None occurs in
  the window.
- *Hidden liquidity.* Hidden orders are not in the book, and their executions change nothing
  visible. Both books are the visible book only.
- *Pre-existing orders.* Orders submitted before 10:00 and cancelled or executed later: the
  messages refer to orders they never introduced. Their size is already in the first row
  of the file, where the rebuild starts.
- *Repeated timestamps.* Many messages share the same time to the nanosecond, so the time
  cannot say which came first. The rebuild follows the order of the rows, which does.
- *Execution granularity.* An aggressive order that hits several resting orders writes one
  row for each. The rebuild applies them one by one, and every intermediate row of the file
  is reproduced.

**Limits:** The alignment of the two files is assumed: a row missing from the middle of both
would go unnoticed. And both books hold one total size per price, so the queue of orders at a
price is neither rebuilt nor compared.

---

## ii. The order flow imbalance and the queue imbalance

> **Asked.** Compute $\mathrm{OFI}_{t,w}$ by Definition 1.4.2 and $I^1_t$ by (1.1.2).
>
> **Found.** $e_n$, $\mathrm{OFI}_{t,w}$ and $I^1_t$ of the package reproduce (1.4.1), (1.4.2)
> and (1.1.2). Over the window the mean $\mathrm{OFI}_{t,w}$ is negative at each of the three
> windows, the mid-price having fallen 41 ticks, and $I^1_t$ averages +0.039.

### ii.1 Which rows are events

The order flow imbalance sums over **events**. In the files, an aggressive order that hits
$K$ resting orders is $K$ rows, $j+1, \dots, j+K$; in `session` it is the single row $j+K$.
By Remark 1.2.6 the session's contribution is that of the transition $j \to j+K$, and

$$e_{j \to j+K} = \sum_{n=j+1}^{j+K} e_n \tag{Remark 1.2.6}$$

holds whenever every fill is at the best price facing the order. It fails only when the
order walks the book, that is, trades at more than one price.

We use `session`: one row, one event. `coarsening_report` counts, for $e_n$ and for
$\mathrm{OFI}$ at the three windows of iii.–v., the rows where the value computed on `session` differs from the
one computed on the rows of the files:

In [ ]:
report = coarsening_report(raw, session).set_index("Statistic")
report.loc[
    ["OrderFlowContribution", "OrderFlowImbalance1", "OrderFlowImbalance10",
     "OrderFlowImbalance60"],
    ["RowsCompared", "RowsDiffering"],
].rename(columns={"RowsCompared": "rows compared", "RowsDiffering": "rows differing"})

Only 15 rows change $e_n$: those of orders that walk the book. A window changes when it
contains one, so the longer the window, the more windows change.

### ii.2 The order flow imbalance

By Definition 1.4.2 the contribution of the $n$-th event is

$$e_n = \mathbf 1_{\{P^b_{T_n} \ge P^b_{T_{n-1}}\}} S^{b,1}_{T_n}
      - \mathbf 1_{\{P^b_{T_n} \le P^b_{T_{n-1}}\}} S^{b,1}_{T_{n-1}}
      - \mathbf 1_{\{P^a_{T_n} \le P^a_{T_{n-1}}\}} S^{a,1}_{T_n}
      + \mathbf 1_{\{P^a_{T_n} \ge P^a_{T_{n-1}}\}} S^{a,1}_{T_{n-1}}. \tag{1.4.1}$$

The cell sets the package's `OrderFlowContribution` against (1.4.1) written out:

In [ ]:
book = session.lobster_book
bid_price, bid_size, ask_price, ask_size = (
    book[column].to_numpy(dtype=float)
    for column in ("BidPrice1", "BidSize1", "AskPrice1", "AskSize1")
)
now, before = slice(1, None), slice(None, -1)
by_definition = np.r_[np.nan, (
    (bid_price[now] >= bid_price[before]) * bid_size[now]
    - (bid_price[now] <= bid_price[before]) * bid_size[before]
    - (ask_price[now] <= ask_price[before]) * ask_size[now]
    + (ask_price[now] >= ask_price[before]) * ask_size[before]
)]
contribution = session.stats["OrderFlowContribution"].to_numpy(dtype=float)
print("e_n of the package equals (1.4.1) on every row:",
      np.array_equal(contribution, by_definition, equal_nan=True))

The two agree on every row.

Summed over a window,

$$\mathrm{OFI}_{t,w} = \sum_{n\,:\,t-w < T_n \le t} e_n. \tag{1.4.2}$$

**The windows:** The assignment fixes no window, so this notebook uses three, $w = 1$, 10 and
60 s, the same in points ii.–v. They lie on either side of the time a best price takes to
move, about 9 s here (iv.): one well below it, one close to it, one well above it.

The cell checks `order_flow_imbalance(Window(w))` against the sum taken directly, at three
windows, and describes it, together with the move of the mid-price over the window. A window
that holds the first row, where $e_n$ is undefined, has no value:

In [ ]:
times = session.lobster_book.index.to_numpy(dtype=float)
rows = np.random.default_rng(0).choice(np.arange(1, len(times)), 1000, replace=False)
WINDOWS = (1.0, 10.0, 60.0)


def by_the_sum(row, window):
    inside = (times[: row + 1] > times[row] - window)
    return contribution[: row + 1][inside].sum()


summary = {}
for window in WINDOWS:
    ofi = session.order_flow_imbalance(Window(window))
    defined = ofi.notna().to_numpy()
    checked = [r for r in rows if defined[r]]
    assert all(by_the_sum(r, window) == ofi.iloc[r] for r in checked)
    summary[f"w = {window:g} s"] = {
        "rows where defined": f"{defined.sum():,}",
        "mean (shares)": f"{ofi.mean():,.0f}",
        "standard deviation (shares)": f"{ofi.std():,.0f}",
        "zero": f"{(ofi == 0).mean():.1%}",
    }
print(f"(1.4.2) checked by direct summation on {len(rows):,} random rows, for each window")
mid_ticks = session.stats["MidPrice"].to_numpy(dtype=float)
print(f"P^m over the window: ${mid_ticks[0] * CENT.tick_size:.3f} to "
      f"${mid_ticks[-1] * CENT.tick_size:.3f}, {mid_ticks[-1] - mid_ticks[0]:+.0f} ticks")
pd.DataFrame(summary)

$\mathrm{OFI}_{t,w}$ is in shares, positive under buying pressure, and its spread grows with
$w$. Its mean is negative in a window where selling dominated: the mid-price fell 41 ticks,
from $\$27.345$ to $\$26.935$.

### ii.3 The queue imbalance

By (1.1.2) with $n = 1$,

$$I^1_t = \frac{S^{b,1}_t - S^{a,1}_t}{S^{b,1}_t + S^{a,1}_t} \in [-1, 1], \tag{1.1.2}$$

positive when the bid is the heavier side. At $n = 1$ the grid level and the reported level
are the same price, so Remark 1.2.4 does not bite.

The cell sets `QueueImbalance1` against the formula, and describes it:

In [ ]:
covered = session.stats["QueueImbalance1Covered"].astype(bool).to_numpy()
imbalance = session.stats["QueueImbalance1"].to_numpy(dtype=float)
queue_by_definition = (bid_size - ask_size) / (bid_size + ask_size)
print("I^1 of the package equals (1.1.2) on every covered row:",
      np.allclose(imbalance[covered], queue_by_definition[covered], rtol=0, atol=1e-12))
pd.Series({
    "rows where defined": f"{covered.sum():,} of {len(covered):,}",
    "mean": f"{np.nanmean(imbalance[covered]):+.3f}",
    "standard deviation": f"{np.nanstd(imbalance[covered]):.3f}",
    "bid heavier (I^1 > 0)": f"{(imbalance[covered] > 0).mean():.1%}",
    "ask heavier (I^1 < 0)": f"{(imbalance[covered] < 0).mean():.1%}",
}).rename("I^1").to_frame()

The two agree on every row. $I^1_t$ is defined on every row, averages +0.039, and the bid is
the heavier side on 54.2% of the rows.

$I^1_t$ is a state of one book, $\mathrm{OFI}_{t,w}$ a flow over a window: points iii.–v.
compare the two.

---

## iii. The mid-price change regressed on the order flow imbalance

> **Asked.** Regress $\Delta P^m_{t,w}$ on $\mathrm{OFI}_{t,w}$ and compare the slope with the
> $\tau/(2\bar S)$ of (1.4.3); report the depth it implies, and how much of the difference
> (1.4.4) accounts for.
>
> **Found.** The slope is $2.4$–$2.6 \times 10^{-5}$ ticks per share, against
> $\tau/(2\bar S) = 3.24 \times 10^{-5}$ with $\bar S = 15{,}420$ shares: the depth it implies,
> 19,635–20,519 shares, is above $\bar S$. (1.4.4) closes 49%, 36% and 27% of the gap at
> $w = 1$, 10 and 60 s.

### iii.1 The identity

Section 1.4 predicts how far the mid-price moves per share of order flow imbalance, and point
iii. checks that prediction on the data. The argument is simple. Under Assumption 1.4.1, the
idealised book of Section 1.4, every queue behind the best quotes holds the same $\bar S$
shares: for the best bid to move up one tick the flow must fill a new queue of $\bar S$ shares,
and for it to move down it must empty one. So $\bar S$ shares of net flow move a quote by one tick $\tau$, and the mid-price, halfway
between the two quotes, by half a tick, $\tau/2$; per share of flow, that is $\tau/(2\bar S)$
(Exercise 1.4.4). In formulas (Proposition 1.4.3),

$$\Delta P^m_{t,w} = P^m_t - P^m_{t-w} = \frac{\tau}{2\bar S}\,\mathrm{OFI}_{t,w} + \varepsilon_{t,w},
\qquad |\varepsilon_{t,w}| \le \tau, \tag{1.4.3}$$

$$\varepsilon_{t,w} = -\frac{\tau}{2\bar S}\Big[\big(S^{b,1}_t - S^{b,1}_{t-w}\big)
- \big(S^{a,1}_t - S^{a,1}_{t-w}\big)\Big], \tag{1.4.4}$$

where $\varepsilon_{t,w}$ is the flow that went into the queues at the best quotes without
moving them (Remark 1.4.5), never more than one tick. The slope of $\Delta P^m_{t,w}$ on
$\mathrm{OFI}_{t,w}$ should therefore be $\tau/(2\bar S)$, and a fitted slope $\hat\beta$
implies the depth $\tau/(2\hat\beta)$, to set against $\bar S$; iii.3 asks how much of the
difference $\varepsilon_{t,w}$ accounts for. Prices are in ticks, so $\tau = 1$.

A real book is not idealised, so $\bar S$ has to be read off it. In Assumption 1.4.1 it is the
size of the queues behind the touch, and in Exercise 1.4.4 and Remark 1.4.27 it is what one
tick of a quote's move is worth in shares: the queue the move uncovers. So $\bar S$ is the mean
size at the second level, the first queue behind the touch. Cont, Kukanov and Stoikov (2014),
who introduced $\mathrm{OFI}$, define depth the same way but measure it at the touch, which
they call only a rough measure; iii.3 shows the result with that choice too. $\bar S$ is read
off the book, not chosen to make the slope fit (1.4.3): otherwise the test would pass by
construction.

### iii.2 The regression

$\Delta P^m_{t,w}$ comes from the course's `backward_change`, which differences the mid-price
against the row *before* the window opens, so that the change and $\mathrm{OFI}_{t,w}$ cover
the same events. The fit has no intercept, as (1.4.3) has none. $\bar S$ is the mean of
$(S^{b,2}_t + S^{a,2}_t)/2$ over the window.

First $\bar S$. It is meant as the size of the first queue behind the touch, so before using
the mean size at the second level the cell checks two things: that the second level is always
one tick behind the first, with no gap in between, and that no level is ever empty, which
would pull the mean down. It also prints the largest mean size of any of the ten levels, used
in the reading below:

In [ ]:
mid = ir.aligned_mid_price(session)
flow = ir.aligned_order_flow(session)
TAU = 1
book = session.lobster_book
LEVEL_MEANS = pd.Series({
    level: (book[f"BidSize{level}"].mean() + book[f"AskSize{level}"].mean()) / 2
    for level in range(1, raw.reported_depth + 1)
})
S_BAR = LEVEL_MEANS[2]
TICK = raw.price_unit
sizes = book[[column for column in book.columns if "Size" in column]]
print("Level 2 one tick behind level 1, on both sides, on every row:",
      bool(((book["BidPrice1"] - book["BidPrice2"] == TICK)
            & (book["AskPrice2"] - book["AskPrice1"] == TICK)).all()))
print("Rows with a size of 0 at some level:", int((sizes == 0).any(axis=1).sum()))
print(f"Largest mean size at any of the {raw.reported_depth} levels: "
      f"{LEVEL_MEANS.max():,.0f} shares, at level {LEVEL_MEANS.idxmax()}")

Both checks hold: the second level is always the queue right behind the touch, and its mean
size is not lowered by empty levels, so $\bar S$ is the mean size at the second level. No
level holds on average more than 16,992 shares, the mean of level 4: the depth any level could
give for $\bar S$ is at most that.

The cell fits $\Delta P^m_{t,w}$ on $\mathrm{OFI}_{t,w}$ through the origin at the three windows,
and sets the slope against $\tau/(2\bar S)$:

In [ ]:
def through_origin(x, y):
    return (x @ y) / (x @ x)


fits, table = {}, {}
for window in WINDOWS:
    change = ir.backward_change(mid, Window(window))
    ofi = ir.backward_sum(flow, Window(window))
    defined = np.isfinite(change) & np.isfinite(ofi)
    x, y = ofi[defined], change[defined]
    slope = through_origin(x, y)
    fits[window] = (change, ofi, defined)
    table[f"w = {window:g} s"] = {
        "slope": f"{slope:.2e}",
        "τ/(2S̄)": f"{TAU / (2 * S_BAR):.2e}",
        "implied depth τ/(2·slope) (shares)": f"{TAU / (2 * slope):,.0f}",
        "S̄, mean size at level 2 (shares)": f"{S_BAR:,.0f}",
        "R²": f"{1 - ((y - slope * x) ** 2).sum() / ((y - y.mean()) ** 2).sum():.3f}",
    }
pd.DataFrame(table)

The slope is about a quarter below $\tau/(2\bar S)$ at every window: the mid-price moves less
per share of imbalance than (1.4.3) predicts. The implied depth, about 20,000 shares, is above
the mean of every level of the book, so no choice of $\bar S$ closes the gap. $R^2$ is high,
0.81 to 0.89: $\mathrm{OFI}_{t,w}$ accounts for most of the variation of the change, with a
smaller slope than (1.4.3) gives. iii.3
looks for why.

First, is the gap more than noise? Neighbouring rows share most of their window, so the cell
takes the standard errors on windows that do not overlap, with the robust variance
(`hac_variance` at lag 0):

In [ ]:
table, apart_fits = {}, {}
for window, (change, ofi, defined) in fits.items():
    apart = [np.searchsorted(mid.times, mid.times[0] + window)]
    while (after := np.searchsorted(mid.times, mid.times[apart[-1]] + window)) < mid.times.size:
        apart.append(after)
    apart = np.array(apart)
    apart = apart[defined[apart]]
    fit = est.ols(ofi[apart, None], change[apart])
    slope, error = fit.coefficients[0], np.sqrt(est.hac_variance(fit, 0)[0, 0])
    apart_fits[window] = (ofi[apart], change[apart], slope)
    table[f"w = {window:g} s"] = {
        "windows": f"{apart.size:,}",
        "slope": f"{slope:.2e}",
        "standard error": f"{error:.1e}",
        "(τ/(2S̄) − slope) / standard error": f"{(TAU / (2 * S_BAR) - slope) / error:.1f}",
    }
pd.DataFrame(table)

On windows that do not overlap the slope is almost the same as before, so the result does not
come from the overlapping rows. Its standard error is small: $\tau/(2\bar S)$ lies 7.0, 8.2 and
5.7 standard errors above the slope at $w$ = 1, 10 and 60 s. The gap is not noise: the
mid-price moves less than (1.4.3) predicts.

The figure shows it for the windows of 10 s: each point is one window, with the fitted slope
and the line $\tau/(2\bar S)$ of (1.4.3):

In [ ]:
x, y, slope = apart_fits[10.0]
ends = np.array([x.min(), x.max()])

ax = axes("Mid-price change against order flow imbalance, non-overlapping windows of 10 s",
          "OFI (thousand shares)", "change of the mid-price (ticks)")
ax.scatter(x / 1000, y, s=14, color=LIGHT_BLUE, alpha=0.6, linewidths=0)
ax.plot(ends / 1000, slope * ends, color=DARK_BLUE, linewidth=2, label="fitted slope")
ax.plot(ends / 1000, TAU / (2 * S_BAR) * ends, color=ORANGE, linewidth=2, linestyle="--",
        label="τ/(2S̄) of (1.4.3)")
ax.legend(frameon=False, loc="upper left")
plt.show()

The fitted line is flatter than that of (1.4.3): the mid-price moves less per share of
imbalance than the identity predicts. On many windows it does not move at all, even with a
large imbalance: that flow went into the queues, which is what iii.3 measures.

### iii.3 What (1.4.4) accounts for

The figure showed many windows where the flow does not move the mid-price. There
$\Delta P^m_{t,w} = 0$ while $\mathrm{OFI}_{t,w} \ne 0$, and $\varepsilon_{t,w}$ cancels the
flow term exactly (Remark 1.4.5): such windows pull the slope towards zero. We read "how much
of the difference (1.4.4) accounts for" as the share of the gap between the slope and
$\tau/(2\bar S)$ that removing $\varepsilon_{t,w}$ closes. Least squares is linear in the
response, so that share is exactly minus the slope of $\varepsilon_{t,w}$ on
$\mathrm{OFI}_{t,w}$, over the gap.

The cell computes $\varepsilon_{t,w}$ by (1.4.4) from the touch sizes at the two ends of each
window, removes it, fits again, and gives the share of the gap closed, with $\bar S$ at the
second level and, for comparison, at the touch:

In [ ]:
bid = ir.AlignedSeries(mid.times, book["BidSize1"].to_numpy(dtype=float), mid.segments)
ask = ir.AlignedSeries(mid.times, book["AskSize1"].to_numpy(dtype=float), mid.segments)


def gap_closed(depth, change, ofi, defined, window):
    """Slopes before and after removing (1.4.4), and the share of the gap closed."""
    residual = -TAU / (2 * depth) * (
        ir.backward_change(bid, Window(window)) - ir.backward_change(ask, Window(window))
    )
    kept = defined & np.isfinite(residual)
    slope = through_origin(ofi[kept], change[kept])
    corrected = through_origin(ofi[kept], (change - residual)[kept])
    return slope, corrected, (corrected - slope) / (TAU / (2 * depth) - slope)


table = {}
for window, fit in fits.items():
    slope, corrected, closed = gap_closed(S_BAR, *fit, window)
    *_, closed_at_touch = gap_closed(LEVEL_MEANS[1], *fit, window)
    table[f"w = {window:g} s"] = {
        "slope of ΔPᵐ on OFI": f"{slope:.2e}",
        "slope of ΔPᵐ − ε on OFI": f"{corrected:.2e}",
        "τ/(2S̄)": f"{TAU / (2 * S_BAR):.2e}",
        "share of the gap closed by (1.4.4)": f"{closed:.0%}",
        "the same, S̄ read at the touch": f"{closed_at_touch:.0%}",
    }
pd.DataFrame(table)

Removing $\varepsilon_{t,w}$ moves the slope towards $\tau/(2\bar S)$ and closes 49%, 36% and
27% of the gap at $w$ = 1, 10 and 60 s, and less with $\bar S$ read at the touch. So the flow
absorbed by the queues explains part of the gap, less of it the longer the window, but not all
of it.

What is left can be pinned down exactly. (1.4.3) assumes that every time a best price moves,
the queue behind it holds $\bar S$ shares. Over a window, $\mathrm{OFI}_{t,w}$ less the change
of the bid size plus the change of the ask size is the sum, over the moves of the best prices,
of the actual size $X_k$ of that queue (the size at the outer of the two prices of the move),
signed by the direction of the move (Section 1.4.4.3, the paragraph after Proposition 1.4.41).
The cell checks this on every row and describes the $X_k$:

In [ ]:
move_bid, move_ask = np.diff(bid_price) / TICK, np.diff(ask_price) / TICK
outer_bid = np.where(move_bid > 0, bid_size[:-1], np.where(move_bid < 0, bid_size[1:], 0.0))
outer_ask = np.where(move_ask > 0, ask_size[1:], np.where(move_ask < 0, ask_size[:-1], 0.0))
print("e_n less the change of S^b plus that of S^a is the signed outer size, on every row:",
      np.array_equal(contribution[1:] - np.diff(bid_size) + np.diff(ask_size),
                     np.sign(move_bid) * outer_bid + np.sign(move_ask) * outer_ask))
moved_bid, moved_ask = move_bid != 0, move_ask != 0
ticks = np.abs(np.r_[move_bid[moved_bid], move_ask[moved_ask]])
outer = np.r_[outer_bid[moved_bid], outer_ask[moved_ask]]
pd.Series({
    "moves of a best price": f"{len(ticks):,}",
    "of which by more than one tick": f"{(ticks > 1).sum():,}",
    "mean of X_k (shares)": f"{outer.mean():,.0f}",
    "median of X_k (shares)": f"{np.median(outer):,.0f}",
    "standard deviation of X_k (shares)": f"{outer.std():,.0f}",
    "S̄ (shares)": f"{S_BAR:,.0f}",
}).rename("moves of the best prices").to_frame()

The identity holds on every row. The best prices move 1,158 times, never by more than one tick,
as (1.4.3) assumes. But the queue met at a move is not always $\bar S$: the $X_k$ average
16,163 shares, close to $\bar S$, with a standard deviation of 7,686, about half of that.

That is the whole of what is left. A move changes $P^m$ by half a tick, so by (1.4.3) and
(1.4.4)

$$\Delta P^m_{t,w} - \frac{\tau}{2\bar S}\,\mathrm{OFI}_{t,w} - \varepsilon_{t,w}
= -\frac{\tau}{2\bar S} \sum_k s_k \big(X_k - n_k \bar S\big),$$

with the sum over the moves inside the window and $n_k = 1$ here. The right-hand side is zero
when every queue met holds exactly $\bar S$, as part iii. of Assumption 1.4.1 asks. So the
gap that $\varepsilon_{t,w}$ leaves comes from the queues at the moves departing from $\bar S$,
and from nothing else.

---

## iv. What the order flow imbalance predicts

> **Asked.** Over a grid of windows $w$ and horizons $h$, report the sign and the strength of
> the relation between $\mathrm{OFI}_{t,w}$ and $P^m_{t+h} - P^m_t$, and say at which $(w, h)$
> it can be read at all.
>
> **Found.** The sign is positive in all twelve cells: buying pressure is followed, on average,
> by a rise of the mid-price. The relation can be read in nine cells, most clearly at
> $w = 1$ s and $h \le 10$ s ($t = 8.3$ and 8.4). It is small: $b$ = 0.016 and 0.060 tick
> there.

### iv.1 The predictive regression

Point iii. put the change of $P^m$ over the window on the flow over the *same* window: by
(1.4.3) that relation is accounting. Here the change is taken *after* $t$, over a horizon $h$:
the flow over $(t-w, t]$, by (1.4.2), and the outcome over $(t, t+h]$, which no identity ties
together. The regression is the empirical counterpart of (1.4.16), Corollary 1.4.17, the
expectation of $P^m_{t+h} - P^m_t$ given the history up to $t$, with the flow entering through
its sign:

$$P^m_{t+h} - P^m_t = a + b\,\operatorname{sign}(\mathrm{OFI}_{t,w}) + u_{t,h}.$$

What Section 1.5 asks of it is the sign, not the magnitude (Remark 1.4.18): the sign of $b$
says whether buying pressure is followed by a rise (continuation) or a fall (reversal).

**Why the sign of the flow:** The diagnostics below print two features of the data. The outcome
is zero on most seconds at the short horizons. And the flow is heavy-tailed: its participation
ratio, the number of seconds its sum of squares is effectively spread over
(`participation_ratio` of the course), is a small fraction of the seconds scored at $w = 1$ s.
A least-squares slope on $\mathrm{OFI}_{t,w}$ itself is then set by a few extreme seconds; the
sign weighs every second alike. The slope on $\mathrm{OFI}_{t,w}$ was fitted first, and it is
reported beside the sign.

**Which seconds are scored:** $t$ runs over one observation a second, the last event of each
second that holds one: the rows of one second hold nearly the same $\mathrm{OFI}_{t,w}$ and
nearly the same outcome, and add rows without adding information. Seconds without an event are
left out, so the forecast is conditional on activity in $(t-1, t]$, which is known at $t$.
Where no event falls in $(t, t+h]$ the outcome is 0: $P^m_{t+h}$ exists, the mid-price of the
last event, and has not moved. `forward_change` leaves it undefined, and dropping those seconds
would select the sample on the activity after $t$, which is what the predictor measures. Every
$(w, h)$ is scored on the same seconds: those where $\mathrm{OFI}_{t,w}$ is defined and the
longest horizon ends inside the window.

The cell picks the seconds and computes the outcome at each horizon:

In [ ]:
HORIZONS = (1.0, 10.0, 60.0, 300.0)
CELLS = [(w, h) for w in WINDOWS for h in HORIZONS]
GRID = pd.MultiIndex.from_tuples(CELLS, names=["w (s)", "h (s)"])
times = mid.times
seconds = np.arange(np.ceil(times[0]), times[-1] + 1.0)
sampled = np.unique(np.searchsorted(times, seconds, side="right") - 1)
sampled = sampled[sampled >= 0]
# forward_change stops at the last row, so near the close a horizon would be cut short.
inside = times[sampled] + max(HORIZONS) <= times[-1]


def change_ahead(h):
    """P^m_{t+h} - P^m_t at the sampled rows, 0 where no event falls in (t, t + h]."""
    change = ir.forward_change(mid, Horizon(h))[sampled]
    last = np.searchsorted(times, times[sampled] + h, side="right") - 1
    quiet = (last == sampled) & np.isfinite(mid.values[sampled])
    return np.where(quiet, 0.0, change)

**How the relation is read:** Outcomes one second apart overlap for $h$ seconds, so their
errors are not independent: the $t$-statistic of $b$ uses the Newey–West variance,
`hac_variance` of the course, at a lag of $h$ seconds. The lag is a choice, so it is
checked. The docstring of the course's `hac_variance` sets the lag from the forecast horizon,
of the order of four times the mean number of events within it, which with one observation a
second is $4h$, and asks to report the result at half and twice the lag as well: a cell whose
reading changed with the lag would rest on that choice rather than on the data. Every table of
iv. and v. prints whether Holm reads the same cells at lags $h/2$, $2h$ and $4h$. $R^2$ is not reported, since the outcome
sits on a lattice of half ticks. The family is the twelve cells $(w, h)$: in each, $b = 0$ is
tested two-sided against the normal distribution, and the twelve $p$-values are corrected
together by Holm's step-down procedure at a family-wise level of 5%. A cell is read when Holm
rejects $b = 0$ there.

The cell defines the fit, the two-sided $p$-value, Holm's procedure, the check at the other
lags, and a helper that writes the tables:

In [ ]:
def newey_west(predictors, y, h, scale=1.0):
    """Slopes of y on an intercept and the predictors, and their Newey–West t at lag scale·h."""
    fit = est.ols(np.column_stack([np.ones(y.size), *predictors]), y)
    error = np.sqrt(np.diag(est.hac_variance(fit, int(np.ceil(scale * h)))))
    return fit.coefficients[1:], fit.coefficients[1:] / error[1:]


def two_sided(t):
    return 2 * stats.norm.sf(np.abs(t))


def holm(p, level=0.05):
    """Where Holm's step-down procedure rejects, family-wise at `level`."""
    p = np.asarray(p, dtype=float)
    rejected = np.zeros(p.size, dtype=bool)
    for rank, index in enumerate(np.argsort(p)):
        if p[index] > level / (p.size - rank):
            break
        rejected[index] = True
    return rejected


# The docstring of the course's hac_variance: a lag of the order of four times the events
# within the horizon, and the result reported at lag / 2 and 2 lag as well.
LAG_SCALES = (0.5, 2.0, 4.0)


def same_at_other_lags(t_at_scale, read):
    """Whether Holm reads the cells `read` again at each lag of LAG_SCALES.

    `t_at_scale(scale)` gives the t-statistics of the family at lag scale·h."""
    verdicts = []
    for scale in LAG_SCALES:
        same = (holm(two_sided(np.asarray(t_at_scale(scale)))) == read).all()
        verdicts.append(f"lag {scale:g}h: {'same' if same else 'different'}")
    return ", ".join(verdicts)


def shown(frame, formats):
    """The frame with each column written in its format."""
    written = frame.apply(lambda column: column.map(formats.get(column.name, "{}").format))
    return written.rename(index=lambda value: f"{value:g}" if isinstance(value, float) else value)

The cell holds the predictor and the outcome on the seconds scored, the only ones used from
here on:

In [ ]:
ofi_sampled = {w: ir.backward_sum(flow, Window(w))[sampled] for w in WINDOWS}
ahead_sampled = {h: change_ahead(h) for h in HORIZONS}
common = inside & np.logical_and.reduce(
    [np.isfinite(series) for series in (*ofi_sampled.values(), *ahead_sampled.values())]
)
dropped = common & ~np.isfinite(ir.forward_change(mid, Horizon(min(HORIZONS)))[sampled])
# From here on the predictor and the outcome are held on the seconds scored only.
ofi_at = {w: x[common] for w, x in ofi_sampled.items()}
ahead = {h: y[common] for h, y in ahead_sampled.items()}

scored = times[sampled[common]]
span = scored[-1] - scored[0]

The diagnostics of iv.1, each with what it is used for. A second where $\mathrm{OFI}_{t,w} = 0$
has sign 0 and stays in the sample; the table counts them at $w = 1$ s:

In [ ]:
spread = session.stats["Spread"].to_numpy(dtype=float)
pd.DataFrame(
    [
        ("seconds scored", f"{common.sum():,}", "the sample of every cell"),
        ("non-overlapping horizons at h = 300 s", f"{span / max(HORIZONS):,.0f}",
         "how little the longest horizon rests on"),
        ("a best price moves once every", f"{(times[-1] - times[0]) / len(ticks):.1f} s",
         "the time scale of the windows and the horizons"),
        ("seconds kept because a quiet horizon counts as no change", f"{dropped.sum():,}",
         "forward_change alone would drop them"),
        ("outcome zero at h = 1 s", f"{np.mean(ahead[1.0] == 0):.1%}",
         "why the flow enters through its sign"),
        ("participation ratio of OFI at w = 1 s",
         f"{est.participation_ratio(ofi_at[1.0]):,.0f} of {common.sum():,}",
         "why the flow enters through its sign"),
        ("OFI zero, so sign 0, at w = 1 s", f"{np.mean(ofi_at[1.0] == 0):.1%}",
         "seconds kept with sign 0"),
        ("spread of one tick, on the session rows",
         f"{np.mean(spread == 1):.1%}, mean {np.nanmean(spread):.2f} ticks",
         "the scale for the size of b"),
    ],
    columns=["diagnostic", "value", "used for"],
).set_index("diagnostic")

The two reasons of iv.1 for using the sign of the flow show in the table. On 96.9% of the
seconds the mid-price does not move in the next second, so the outcome is almost always zero.
And the flow is dominated by a few extreme seconds: at $w = 1$ s, out of 9,517 seconds, a
handful carry as much of its variation as 220 ordinary ones would, so a slope on
$\mathrm{OFI}_{t,1}$ itself would be set by them. The sign gives every second the same weight.
On 9.7% of the seconds $\mathrm{OFI}_{t,1}$ is zero; they stay in the sample, with sign 0.

The grid has the windows in rows and the horizons in columns. Each cell gives $b$ in ticks,
which carries the sign and the size of the relation, then its $t$ in brackets, and a ✓ where
Holm reads it. Above the table the cell prints, for comparison, the slope on
$\mathrm{OFI}_{t,w}$ itself:

In [ ]:
rows = []
for w, h in CELLS:
    x, y = ofi_at[w], ahead[h]
    (b,), (t,) = newey_west([np.sign(x)], y, h)
    (slope,), (t_ofi,) = newey_west([x], y, h)
    rows.append({"b": b, "t": t, "p": two_sided(t), "slope on OFI": slope, "t on OFI": t_ofi})
grid = pd.DataFrame(rows, index=GRID)
grid.insert(3, "Holm", np.where(holm(grid["p"]), "read", ""))
print("cells read at the other lags: " + same_at_other_lags(
    lambda scale: [newey_west([np.sign(ofi_at[w])], ahead[h], h, scale)[1][0] for w, h in CELLS],
    (grid["Holm"] == "read").to_numpy()))

print(f"slope on OFI itself: positive in {(grid['slope on OFI'] > 0).sum()} of {len(grid)} "
      f"cells, t from {grid['t on OFI'].min():.1f} to {grid['t on OFI'].max():.1f}")
cells = grid.apply(lambda r: f"{r['b']:+.3f} ({r['t']:.1f})" + (" ✓" if r["Holm"] else ""),
                   axis=1)
cells.unstack("h (s)").rename(
    index=lambda w: f"w = {w:g} s", columns=lambda h: f"h = {h:g} s"
).rename_axis(index=None, columns=None)

### iv.2 Reading the grid

The assignment asks three things of the grid: the sign of the relation, its strength, and where
it can be read at all.

**Sign:** Positive in every cell: in this window, buying pressure over the last $w$ seconds is
followed, on average, by a rise of the mid-price. The slope on $\mathrm{OFI}_{t,w}$ itself has
the same sign in every cell.

**Strength:** Small. $b$ is in ticks: 0.016 at $(1\text{ s}, 1\text{ s})$ and 0.060 at
$(1\text{ s}, 10\text{ s})$, against a spread of one tick on 77.6% of the rows, 1.22 ticks on
average, so the expected mid-price moves by a few hundredths of a tick. Measured by $t$, the
relation is strongest at the shortest window and the shorter horizons, $t = 8.3$ and 8.4 at
$w = 1$ s and $h = 1$ and 10 s, and weakens as the window or the horizon grows. The slope on
$\mathrm{OFI}_{t,w}$ itself is weaker, its $t$ never above 2.8, as iv.1 expected.

**Where it can be read:** By the criterion of iv.1, in nine cells of the twelve:

- at $w = 1$ s up to $h = 60$ s and, only just, at 300 s;
- at $w = 10$ s up to $h = 60$ s;
- at $w = 60$ s up to $h = 10$ s.

It cannot be read at $(10\text{ s}, 300\text{ s})$, $(60\text{ s}, 60\text{ s})$ or
$(60\text{ s}, 300\text{ s})$, the bottom right of the table. The same nine are read with the
Newey–West lag halved or doubled. At $4h$, the order the docstring suggests, the weakest of
the nine, $(1\text{ s}, 300\text{ s})$, is no longer read. It is read by Holm at its last step, on about 35 non-overlapping
horizons, which is little for the Newey–West variance to rest on. At $w = 60$ s, Remark 1.2.6
adds that about one window in ten of the coarsened session has a different
$\mathrm{OFI}_{t,w}$ from the one on the rows of the files (ii.1).

**Compared with the notes:** Two points of Section 1.4 bear on the result.

- *How short a horizon.* Remark 1.4.24 expects no forecast at horizons much shorter than the
  time a best price takes to move, because there the flow goes into the queues. Here a best
  price moves about once every 9 s, yet the relation already shows at $h = 1$ s.
- *Why positive.* In the model the sign of the forecast depends on how the flow feeds itself,
  through $\theta$ of (1.4.13): positive when events beget more events pushing the same way,
  as in the trending specification of the course, and not in the resilient one. The positive
  $b$ here is consistent with a trending regime, but it does not prove it: $\theta$ is not
  estimated, and in the course notebook `regressing-the-mid-on-the-imbalances` (sections 5
  and 6), on simulated flow with no such feeding ($\rho = 0$), the sign comes out negative.

---

## v. What the queue imbalance predicts

> **Asked.** Repeat iv. with $I^1_t$ in place of $\mathrm{OFI}_{t,w}$. Remark 1.5.1 does not
> bind here, the flow being traded, so the comparison is symmetric; say what that changes.
>
> **Found.** $I^1_t$ alone is read at $h = 1$, 10 and 60 s ($t = 11.5$, 10.2, 3.1), with a
> positive sign. Together with the flow, $\mathrm{OFI}_{t,w}$ at its sign and $I^1_t$ at its
> level, $I^1_t$ adds in all nine cells up to 60 s, and the flow adds in three. Symmetry means
> the result counts at face value, whichever predictor it favours.

### v.1 The queue imbalance alone

Point iv. with $I^1_t$ of (1.1.2) in place of the flow:

$$P^m_{t+h} - P^m_t = a + c\,I^1_t + u_{t,h}.$$

The two predictors enter differently, each for its own reason.

| | $\mathrm{OFI}_{t,w}$ | $I^1_t$ |
| --- | --- | --- |
| how it enters | through its sign, as in iv. | at its level |
| why | it is heavy-tailed | it lies in $[-1, 1]$ and has no tail to tame |
| window $w$ | three windows | none: it is the state of the book at $t$, not a sum over a window |
| family, alone | the twelve cells $(w, h)$ | the four horizons |
| Remark 1.2.6 | bears on it (iv.2) | concerns sums over windows and does not bear on it |

$I^1_t$ alone is tested two-sided and corrected by Holm at 5% over its four horizons. The
seconds, the Newey–West variance and its check at the other lags are those of iv.

The cell checks that $I^1_t$ is defined on every second scored in iv., and fits it alone at
each horizon. The table gives $c$, in ticks per unit of $I^1_t$, its $t$ in brackets, and a ✓
where Holm reads it; above it the cell prints the check at the other lags:

In [ ]:
queue = np.where(
    session.stats["QueueImbalance1Covered"].astype(bool).to_numpy(),
    session.stats["QueueImbalance1"].to_numpy(dtype=float), np.nan,
)[sampled][common]
print(f"I^1 is defined on {np.isfinite(queue).sum():,} of the {common.sum():,} seconds of iv.")
assert np.isfinite(queue).all()

rows = []
for h in HORIZONS:
    (c,), (t,) = newey_west([queue], ahead[h], h)
    rows.append({"c": c, "t": t, "p": two_sided(t)})
queue_grid = pd.DataFrame(rows, index=pd.Index(HORIZONS, name="h (s)"))
queue_grid["Holm"] = np.where(holm(queue_grid["p"]), "read", "")
print("horizons read at the other lags: " + same_at_other_lags(
    lambda scale: [newey_west([queue], ahead[h], h, scale)[1][0] for h in HORIZONS],
    (queue_grid["Holm"] == "read").to_numpy()))
row = queue_grid.apply(lambda r: f"{r['c']:+.3f} ({r['t']:.1f})" + (" ✓" if r["Holm"] else ""),
                       axis=1)
row.rename(lambda h: f"h = {h:g} s").rename("I¹").to_frame().T.rename_axis(columns=None)

**Sign:** Positive at every horizon: in this window, a heavier bid is followed, on average, by
a rise of the mid-price.

**Strength:** $c$ is in ticks per unit of $I^1_t$, which runs from $-1$ to 1: going from a
balanced touch to one with all the size on the bid raises the expected mid-price by 0.076 tick
over the next second and 0.421 over the next ten. Measured by $t$, 11.5 at $h = 1$ s and 10.2
at 10 s, it is stronger than any cell of iv.

**Where it can be read:** At $h = 1$, 10 and 60 s, not at 300 s; the same horizons at
the other three lags.

The figure sets the two predictors side by side, each alone: the $t$-statistic of
$\operatorname{sign}(\mathrm{OFI}_{t,w})$ from iv., one line for each window, and that of
$I^1_t$. A filled marker is a cell that Holm reads in its own family:

In [ ]:
ax = axes("t-statistic of each predictor alone, by horizon", "horizon h", "t-statistic")
places = np.arange(len(HORIZONS))
series = [(f"sign(OFI), w = {w:g} s", grid.xs(w, level="w (s)"), colour)
          for w, colour in zip(WINDOWS, (LIGHT_BLUE, BLUE, DARK_BLUE))]
series.append(("I¹", queue_grid, ORANGE))
for name, table, colour in series:
    read = (table["Holm"] == "read").to_numpy()
    ax.plot(places, table["t"], color=colour, linewidth=2, label=name)
    ax.scatter(places[read], table["t"][read], s=45, color=colour, zorder=3)
    ax.scatter(places[~read], table["t"][~read], s=45, facecolors="white", edgecolors=colour,
               linewidths=2, zorder=3)
ax.axhline(0, color=MUTED, linewidth=0.8)
ax.set_xticks(places)
ax.set_xticklabels([f"{h:g} s" for h in HORIZONS])
ax.legend(frameon=False)
plt.show()

At $h = 1$ and 10 s the line of $I^1_t$ lies above those of
$\operatorname{sign}(\mathrm{OFI}_{t,w})$ at every $w$. At 60 s it falls below those of
$w = 1$ and 10 s, and at 300 s only $\operatorname{sign}(\mathrm{OFI}_{t,1})$ is read.

### v.2 The two together

Each predictor alone does not say whether one adds to the other. The two together,

$$P^m_{t+h} - P^m_t = a + b\,\operatorname{sign}(\mathrm{OFI}_{t,w}) + c\,I^1_t + u_{t,h},$$

ask two questions:

- does the flow add over the state of the touch ($b \ne 0$)?
- does the state of the touch add over the flow ($c \ne 0$)?

Each is a family of its own, the twelve cells $(w, h)$ of its coefficient, corrected by Holm on
its own at 5%, because each answers one of the two questions.

The cell fits the two together in every cell. The table has a block for each coefficient, with
the windows in rows and the horizons in columns: the coefficient, its $t$ in brackets and a ✓
where Holm reads it. Above it the cell prints the check at the other lags:

In [ ]:
rows = []
for w, h in CELLS:
    (b, c), (t_b, t_c) = newey_west([np.sign(ofi_at[w]), queue], ahead[h], h)
    rows.append({"b": b, "t of b": t_b, "p of b": two_sided(t_b),
                 "c": c, "t of c": t_c, "p of c": two_sided(t_c)})
joint = pd.DataFrame(rows, index=GRID)
joint["Holm, b"] = np.where(holm(joint["p of b"]), "read", "")
joint["Holm, c"] = np.where(holm(joint["p of c"]), "read", "")
both_at = {scale: [newey_west([np.sign(ofi_at[w]), queue], ahead[h], h, scale)[1]
                   for w, h in CELLS] for scale in LAG_SCALES}
for k, name in enumerate(("b", "c")):
    print(f"cells of {name} read at the other lags: " + same_at_other_lags(
        lambda scale: [t[k] for t in both_at[scale]],
        (joint[f"Holm, {name}"] == "read").to_numpy()))

blocks = {}
for name, label, digits in (("b", "b, sign(OFI)", 4), ("c", "c, I¹", 3)):
    written = joint.apply(lambda r: f"{r[name]:+.{digits}f} ({r[f't of {name}']:.1f})"
                          + (" ✓" if r[f"Holm, {name}"] else ""), axis=1)
    blocks[label] = written.unstack("h (s)").rename(
        index=lambda w: f"w = {w:g} s", columns=lambda h: f"h = {h:g} s")
pd.concat(blocks).rename_axis(index=[None, None], columns=None)

With $\mathrm{OFI}_{t,w}$ at its sign and $I^1_t$ at its level, as in v.1:

- $c$ is read in all nine cells up to $h = 60$ s, at values close to those of $I^1_t$ alone;
- $b$ is read in three: at $(1\text{ s}, 1\text{ s})$ and $(1\text{ s}, 10\text{ s})$,
  positive, $t = 6.3$ and 5.1; and at $(60\text{ s}, 1\text{ s})$, negative, $t = -3.4$;
- both readings are the same at the other three lags.

So, with these two transformations, the state of the touch adds over the flow in every cell up
to 60 s, and the flow of the last second adds over the state at the two shortest horizons
only.

The negative $b$ at $(60\text{ s}, 1\text{ s})$ is one cell, found with these two
transformations, against the positive sign of the others, and stays an observation.

### v.3 What symmetry changes

The assignment asks what changes when Remark 1.5.1 does not bind, as here, the flow
being traded.

- *In the generated setting*, Remark 1.5.1 reads such a comparison asymmetrically. Whatever
  $I^1_t$ predicts there is mechanical: a depleted queue is one market order from clearing. So
  a result in favour of $I^1_t$ is the stronger finding, and one in favour of $\mathrm{OFI}$
  says little about a traded market.
- *Here the flow is traded*, and the asymmetry goes. What $I^1_t$ predicts may be mechanical,
  or it may be information, a thin side being thin because those posting on it inferred
  something. This notebook does not tell the two apart, and the comparison does not need them
  to.

The result counts at face value: in this window the state of the touch predicts the next moves
of the mid-price and adds over the flow up to 60 s, while, with the flow at its sign and the
state at its level, the flow adds over the state only at the two shortest horizons. Had the
flow come out ahead, that would have counted equally.

---

## Summary

INTC, 21 June 2012, 10:00–13:00. Every statement holds in this window only.

**i. The session, and the reconciliation of the book**

- Rebuilt from the messages alone, the book agrees with the orderbook file on all ten levels
  for the first 63 rows only, and at the first level on 59.58% of the rows.
- Every difference the comparison can see comes from the file showing ten levels only: with a
  repair that reads 579 levels from the file, every row agrees.

**ii. The order flow imbalance and the queue imbalance**

- $e_n$, $\mathrm{OFI}_{t,w}$ and $I^1_t$ of the package reproduce (1.4.1), (1.4.2) and (1.1.2).
- The mid-price falls 41 ticks over the window, and the mean $\mathrm{OFI}_{t,w}$ is negative at
  all three windows. $I^1_t$ averages +0.039.

**iii. The mid-price change regressed on the order flow imbalance**

- The slope is $2.4$–$2.6 \times 10^{-5}$ ticks per share, below
  $\tau/(2\bar S) = 3.24 \times 10^{-5}$, with $\bar S = 15{,}420$ shares read at level 2.
- On windows that do not overlap, $\tau/(2\bar S)$ lies 5.7 to 8.2 standard errors above the
  slope.
- The implied depth, 19,635–20,519 shares, is above the mean size of every level of the book.
- (1.4.4) closes 49%, 36% and 27% of the gap at $w = 1$, 10 and 60 s. The rest comes from the
  queues met at the moves departing from $\bar S$.

**iv. What the order flow imbalance predicts**

- *Sign:* Positive in all twelve cells: buying pressure is followed, on average, by a rise of
  the mid-price.
- *Strength:* Small: $b$ = 0.016 and 0.060 tick at $w = 1$ s and $h = 1$ and 10 s, where the
  relation is clearest ($t = 8.3$ and 8.4).
- *Where it can be read:* In nine cells of twelve, the same nine at lags $h/2$ and $2h$; at
  $4h$, $(1\text{ s}, 300\text{ s})$ drops out.

**v. What the queue imbalance predicts**

- Alone, $I^1_t$ has a positive sign and is read at $h = 1$, 10 and 60 s ($t = 11.5$, 10.2,
  3.1); at $h = 1$ and 10 s it is stronger than any cell of iv.
- Together with the flow, $I^1_t$ adds in all nine cells up to 60 s, and the flow in three.
- The flow being traded, the comparison is symmetric: the result counts at face value,
  whichever predictor it favours. The notebook does not tell a mechanical forecast from an
  informational one.